# Face Mask Detection with CNN

## Overview
Advance deep learning and data engineering expertise through projects that apply neural networks and data processing techniques to real-world problems. This includes developing data aggregation and visualization modules for healthcare analytics, building predictive models for banking retention, applying computer vision for mask detection, leveraging hybrid models for sentiment analysis, and using autoencoders for medical image denoising. These projects emphasize model design, data handling, and evaluation to strengthen technical proficiency in supervised and unsupervised learning.

## Project Statement
Develop CNN model across diverse domains through a hands-on project that addresses distinct real-world challenges involving image data, emphasizing model design, data preprocessing, and performance evaluation. The projects promote practical understanding of CNNs through industry-relevant problem-solving using neural networks.

## Task: Detect humans wearing face masks

### Requirements:

1. **Load and preprocess the image datasets**
   - Prepare Training and Testing Datasets

2. **Develop and train CNN model**
   - Ensure detailed discussion and well thought out reasoning about Model architecture
   - Include your reasoning for:
	 - Layers
	 - Inputs (up to you to research and decide) and Output Layers
	 - Activation Functions
	 - Hyperparameters
	 - Training and Validation Metrics
	 - Loss Function

3. **Use callbacks and early stopping for efficient optimization**
   - Decide on a reasonable metric to monitor
   - Describe why this is the metric that you should monitor
   - Provide evidence / describe why your early stopping worked

4. **Evaluate and compare model performance**
   - Test Data
   - Visualize the predictions
   - Include the image along with the True & Predicted labels
   - Determine the best-performing model

### Extra Credit (Optional):
- Include Data Augmentation Procedures
- Train a Finalized Model on ALL your data
- Use StreamLit - Drawable Canvas to get live model inference via drawing


In [15]:
# Install software
%pip install optuna optuna-dashboard Pillow --quiet

Note: you may need to restart the kernel to use updated packages.


In [16]:
# Imports
# Standard library imports
from pathlib import Path
import glob

# Third-party imports
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim
import optuna
from optuna import Trial

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = 'cpu'
print('Device: ', device)

Device:  cpu


In [17]:
# Hyperparameters lists of potential values to optimize over
num_classes = 1
learning_rate_l = [1e-4, 1e-3, 1e-2]
initial_filters_l = [16, 32, 64]
n_conv_blocks_l = [2, 3, 4]
batch_size_l = [64, 128, 256]
fc_units_1_l = [16, 32, 64]
fc_units_2_l = [16, 32, 64]
dropout_rate_l = [0.3, 0.4, 0.5]


In [18]:
# Unzip our data from zip file if data dir does not exist
data_path = Path('data')
if not data_path.is_dir():
    data_path.mkdir(parents=True, exist_ok=True)
    import zipfile
    with zipfile.ZipFile('data.zip', 'r') as zip_ref:
        zip_ref.extractall('.')

# combine the image datasets from the extracted folders into labelled a pandas dataframe with the columns 'image_matrix' and 'label'
with_mask_img_path = data_path / 'with_mask'
without_mask_img_path = data_path / 'without_mask'

filenames_with_mask = glob.glob(str(with_mask_img_path / '*.jpg'))
filenames_without_mask = glob.glob(str(without_mask_img_path / '*.jpg'))
# Get files as PIL images and convert to RGB matrices
with_mask_img_mtx_list = [Image.open(img).convert('RGB') for img in filenames_with_mask]
without_mask_img_mtx_list = [Image.open(img).convert('RGB') for img in filenames_without_mask]

# Attach labels to the dataframe (1 for with_mask, 0 for without_mask)
with_masks_df = pd.DataFrame({'image_matrix': with_mask_img_mtx_list, 'with_mask': 0.94})
without_masks_df = pd.DataFrame({'image_matrix': without_mask_img_mtx_list, 'with_mask': 0.01})
masks_df = pd.concat([with_masks_df, without_masks_df], ignore_index=True)

del with_masks_df
del without_masks_df

print(masks_df.head())
print(masks_df.shape)



f:\AI and Machine Learning course\FSA_devops\.venv-13\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


                                        image_matrix  with_mask
0  <PIL.Image.Image image mode=RGB size=525x350 a...       0.94
1  <PIL.Image.Image image mode=RGB size=135x218 a...       0.94
2  <PIL.Image.Image image mode=RGB size=86x105 at...       0.94
3  <PIL.Image.Image image mode=RGB size=310x403 a...       0.94
4  <PIL.Image.Image image mode=RGB size=121x204 a...       0.94
(7553, 2)


In [19]:

# Create a custom torch dataset from the dataframe, the transformer and the feature and label column names
# class CustomDataset(torch.utils.data.Dataset):
#     def __init__(self, df, transform, feat, label):
#         self.df = df
#         self.transform = transform
#         self.feat = feat
#         self.label = label

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         img_mtx = self.df.iloc[idx][self.feat]
#         label = self.df.iloc[idx][self.label]
#         img = Image.fromarray(img_mtx)
#         if self.transform:
#             img = self.transform(img)
#         return img, label
    



In [20]:
# Define the transformer
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

In [21]:
# Split dataset into training and testing sets
# Create training, validation and testing tensors
X_train_full = torch.stack([transform(img) for img in masks_df['image_matrix']]).to(device)
y_train_full = torch.tensor([label for label in masks_df['with_mask']]).to(device)
y_train_full = y_train_full.reshape(-1, 1)

print('Size of training features: ', X_train_full.shape)
print('Head of training features: ', X_train_full[:5])
print('Size of training labels: ', y_train_full.shape)
print('Head of training labels: ', y_train_full[:5])

# Split training data into train and validation sets (80/20 split)
n_train = int(0.8 * len(X_train_full))
indices = torch.randperm(len(X_train_full))

X_train = X_train_full[indices[:n_train]]
y_train = y_train_full[indices[:n_train]]
X_test = X_train_full[indices[n_train:]]
y_test = y_train_full[indices[n_train:]]

# Create TensorDatasets and DataLoaders
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
test_dataset = torch.utils.data.TensorDataset(X_test, y_test)

# print sizes
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Size of training features:  torch.Size([7553, 3, 32, 32])
Head of training features:  tensor([[[[-1.0000, -1.0000, -0.4353,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -0.4196,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -0.3725,  ..., -0.7725, -1.0000, -0.8745],
          ...,
          [-0.2235, -0.3412, -0.7490,  ..., -0.9608, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -0.8275, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -0.7490, -1.0000, -1.0000]],

         [[-1.0000, -1.0000, -0.4510,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -0.4353,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -0.4118,  ..., -0.7882, -1.0000, -0.9216],
          ...,
          [-0.1922, -0.2941, -0.6157,  ..., -0.9765, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -0.8667, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -0.8275, -1.0000, -1.0000]],

         [[-1.00

In [29]:
# Custom loss function using nn.Module as a base class
class CustomLoss(nn.Module):
    def __init__(self, 
                 apply_class_balancing=False,
                 alpha=0.25,
                 gamma=2.0,
                 from_logits=False,
                 label_smoothing=0.0,
                 axis=-1,
                 reduction='sum', # 'none', 'mean', 'sum'
                 weight=None
                 ):
        super().__init__()
        self.apply_class_balancing = apply_class_balancing
        self.alpha = alpha
        self.gamma = gamma
        self.from_logits = from_logits
        self.label_smoothing = label_smoothing
        self.axis = axis
        self.reduction = reduction
        self.weight = weight
        self.name = 'binary_focal_crossentropy'

    def forward(self, y_pred, y_true):
        
        ce_base = nn.BCELoss(reduction=self.reduction,
                             weight=self.weight,
                             # label_smoothing=self.label_smoothing) # Not needed for Binary Cross-Entropy Loss in PyTorch, as it does not support label smoothing directly
                             )
        p_t = ce_base(y_pred, y_true)
        # CE(p_t) = − log(p_t)
        # FL(p_t) = −(1 − p_t)γ * log(p_t)
        # or equivalently:
        # FL(p_t) = (1 - p_t) ** gamma * CE(p_t)
        if self.gamma != 0:
            focal_loss = (self.alpha * (1 - p_t) ** self.gamma) * p_t
            return focal_loss
        else:
            # If gamma is 0, focal loss is just cross-entropy loss
            return p_t



In [30]:
# Create the CNN using sequential layers

def create_cnn(n_conv_blocks: int, dropout_rate: float, initial_filters: int, fc_units_1: int, fc_units_2: int, num_classes: int) -> nn.Sequential:
    '''Create a CNN with configurable architecture.
    
    Args:
        n_conv_blocks: Number of convolutional blocks (1-4)
        dropout_rate: Dropout probability
        fc_units_1: Number of units in the first fully connected layer
        fc_units_2: Number of units in the second fully connected layer
        initial_filters: Number of filters in the first convolutional block
        num_classes: Number of output classes
    
    Returns:
        nn.Sequential model
    '''
    layers = []
    in_channels = 3  # RGB input
    current_size = 32  # Input image size
    
    for block_idx in range(n_conv_blocks):
        out_channels = initial_filters * (2 ** block_idx)
        
        # Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU -> Pool -> Dropout
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        
        layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        
        layers.append(nn.MaxPool2d(2, 2))
        layers.append(nn.Dropout(dropout_rate))
        
        in_channels = out_channels
        current_size //= 2
    
    # Calculate flattened size
    final_channels = initial_filters * (2 ** (n_conv_blocks - 1))
    flattened_size = final_channels * current_size * current_size
    
    # Classifier (3 fully connected layers)
    layers.append(nn.Flatten())
    layers.append(nn.Linear(flattened_size, fc_units_1))
    layers.append(nn.ReLU())
    layers.append(nn.Dropout(dropout_rate))
    layers.append(nn.Linear(fc_units_1, fc_units_2))
    layers.append(nn.ReLU())
    layers.append(nn.Dropout(dropout_rate))
    layers.append(nn.Linear(fc_units_2, num_classes))
    layers.append(nn.Sigmoid())  # For binary classification, use sigmoid activation at the end
    
    return nn.Sequential(*layers)


In [31]:
# Use our function to create the CNN
model1 = create_cnn(n_conv_blocks=3, dropout_rate=0.4, initial_filters=32, fc_units_1=64, fc_units_2=32, num_classes=1)
print(model1)

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU()
  (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (7): Dropout(p=0.4, inplace=False)
  (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU()
  (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (13): ReLU()
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (15): Dropout(p=0.4, inplace=False)
  (16): Conv2d(64, 128, kernel_size=(3, 3), strid

In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    epochs: int = 10,
    print_every: int = 1,
    device: torch.device = None
) -> dict[str, list[float]]:
    '''Training loop for PyTorch classification model.
    
    Args:
        device: If provided, moves batches to this device on-the-fly.
                If None, assumes data is already on the correct device.
    '''
    
    history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'train_correct': [], 'val_correct': []}

    for epoch in range(epochs):

        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        print('Train Loader: ', train_loader)
        print('Val Loader: ', val_loader)
        for images, labels in train_loader:
            
            # Move batch to device if specified
            if device is not None:
                images, labels = images.to(device), labels.to(device)

            # Forward pass
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Track metrics
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            history['train_correct'].append(correct)
        # Calculate training metrics
        train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct / total

        # Validation phase
        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:
                
                # Move batch to device if specified
                if device is not None:
                    images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs.data, labels)

                val_running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                history['val_correct'].append(val_correct)
        val_loss = val_running_loss / len(val_loader)
        val_accuracy = 100 * val_correct / val_total

        # Record metrics
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_accuracy'].append(train_accuracy)
        history['val_accuracy'].append(val_accuracy)

        # Print progress
        if (epoch + 1) % print_every == 0 or epoch == 0:

            print(
                f'Epoch {epoch+1}/{epochs} - ' +
                f'loss: {train_loss:.4f} - ' +
                f'accuracy: {train_accuracy:.2f}% - ' +
                f'val_loss: {val_loss:.4f} - ' +
                f'val_accuracy: {val_accuracy:.2f}%'
            )

    print('\nTraining complete.')

    return history

In [33]:
def evaluate_model(
    model: nn.Module,
    test_loader: DataLoader,
    device: torch.device = None
) -> tuple[float, np.ndarray, np.ndarray]:
    '''Evaluate model on test set.
    
    Args:
        device: If provided, moves batches to this device on-the-fly.
                If None, assumes data is already on the correct device.
    '''

    model.eval()
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for images, labels in test_loader:
            
            # Move batch to device if specified
            if device is not None:
                images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * correct / total
    return accuracy, np.array(all_predictions), np.array(all_labels)

In [34]:
# Use optuna to find the best hyperparameters for our model

def objective(trial: Trial) -> float:
    # Suggest hyperparameters for this trial
    learning_rate = trial.suggest_categorical('learning_rate', learning_rate_l)
    initial_filters = trial.suggest_categorical('initial_filters', initial_filters_l)
    n_conv_blocks = trial.suggest_categorical('n_conv_blocks', n_conv_blocks_l)
    batch_size = trial.suggest_categorical('batch_size', batch_size_l)
    fc_units_1 = trial.suggest_categorical('fc_units_1', fc_units_1_l)
    fc_units_2 = trial.suggest_categorical('fc_units_2', fc_units_2_l)
    dropout_rate = trial.suggest_categorical('dropout_rate', dropout_rate_l)

    # Create the model with the suggested hyperparameters
    model = create_cnn(n_conv_blocks=n_conv_blocks, 
                       dropout_rate=dropout_rate, 
                       initial_filters=initial_filters, 
                       fc_units_1=fc_units_1, 
                       fc_units_2=fc_units_2, 
                       num_classes=num_classes).to(device)

    print(model)
    # Define the optimizer and loss function
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = CustomLoss()
    #criterion = nn.BCELoss()
    # Define dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    

    # Train the model for a few epochs
    num_epochs = 20
    history = train_model(model, train_loader, test_loader, criterion, optimizer, epochs=num_epochs, device=device)
    print(history)
    
    # Evaluate the model on the test set
    accuracy, _, _ = evaluate_model(model, test_loader, device=device)
    return accuracy

In [35]:
# Use optuna function to optimize the hyperparameters and view results in the optuna dashboard
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)
print("Best hyperparameters:", study.best_params)

[I 2026-02-09 20:49:04,011] A new study created in memory with name: no-name-7f1b165a-5f24-48a5-95aa-daa98712b5ca


Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU()
  (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (7): Dropout(p=0.3, inplace=False)
  (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU()
  (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (13): ReLU()
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (15): Dropout(p=0.3, inplace=False)
  (16): Conv2d(64, 128, kernel_size=(3, 3), strid

[I 2026-02-09 20:51:54,151] Trial 0 finished with value: 0.0 and parameters: {'learning_rate': 0.01, 'initial_filters': 32, 'n_conv_blocks': 3, 'batch_size': 256, 'fc_units_1': 16, 'fc_units_2': 16, 'dropout_rate': 0.3}. Best is trial 0 with value: 0.0.


Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU()
  (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (7): Dropout(p=0.5, inplace=False)
  (8): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU()
  (11): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (13): ReLU()
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (15): Dropout(p=0.5, inplace=False)
  (16): Conv2d(32, 64, kernel_size=(3, 3), stride

[I 2026-02-09 20:53:29,839] Trial 1 finished with value: 0.0 and parameters: {'learning_rate': 0.0001, 'initial_filters': 16, 'n_conv_blocks': 4, 'batch_size': 256, 'fc_units_1': 32, 'fc_units_2': 16, 'dropout_rate': 0.5}. Best is trial 0 with value: 0.0.


...

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU()
  (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (7): Dropout(p=0.5, inplace=False)
  (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU()
  (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (13): ReLU()
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (15): Dropout(p=0.5, inplace=False)
  (16): Conv2d(64, 128, kernel_size=(3, 3), strid

[I 2026-02-09 21:25:52,862] Trial 8 finished with value: 0.0 and parameters: {'learning_rate': 0.0001, 'initial_filters': 32, 'n_conv_blocks': 4, 'batch_size': 128, 'fc_units_1': 32, 'fc_units_2': 16, 'dropout_rate': 0.5}. Best is trial 0 with value: 0.0.


Sequential(
  (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU()
  (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (7): Dropout(p=0.3, inplace=False)
  (8): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU()
  (11): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (13): ReLU()
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (15): Dropout(p=0.3, inplace=False)
  (16): Conv2d(32, 64, kernel_size=(3, 3), stride

[I 2026-02-09 21:27:40,013] Trial 9 finished with value: 0.0 and parameters: {'learning_rate': 0.0001, 'initial_filters': 16, 'n_conv_blocks': 4, 'batch_size': 64, 'fc_units_1': 16, 'fc_units_2': 32, 'dropout_rate': 0.3}. Best is trial 0 with value: 0.0.


Best hyperparameters: {'learning_rate': 0.01, 'initial_filters': 32, 'n_conv_blocks': 3, 'batch_size': 256, 'fc_units_1': 16, 'fc_units_2': 16, 'dropout_rate': 0.3}


### Final Notes
- I did not get finished with this project and ended up getting hung on shaping errors and debugging the loss function I made
- I wanted to pass this on for something and to show some work and I will just end up continuing to work on it later
- Through a few of my testing iterations I was able to get that loss function working, and it does not seem difficult to switch between a multi-class classifier version and a binary classifier
